# 🛠️ Урок 22 — Улучшаем модель и делаем интерфейс

Сегодня: улучшить модель, честно оценить, обернуть в сайт и подготовить рассказ.

> ★ Помни про рубрику на 20 баллов: baseline, Pipeline, метрика, интерфейс, README, защита.

## Шаг 0 · Твоя модель с урока 21
Запусти свой старт (задача → данные → Pipeline). Пример — пингвины, **замени на свой датасет**.

In [ ]:
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
df = sns.load_dataset('penguins').drop(columns=['sex'])   # 👈 пример; sex всегда убираем
df = df.dropna(subset=['species'])
num = ['bill_length_mm','bill_depth_mm','flipper_length_mm','body_mass_g']  # 👈 свои числовые
cat = ['island']                                           # 👈 свои категориальные
X = df[num+cat]; y = df['species']                         # 👈 свой target
X_tr,X_te,y_tr,y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
prep = ColumnTransformer([
    ('num', Pipeline([('i',SimpleImputer(strategy='median')),('s',StandardScaler())]), num),
    ('cat', Pipeline([('i',SimpleImputer(strategy='most_frequent')),('o',OneHotEncoder(handle_unknown='ignore'))]), cat)])
model = Pipeline([('prep',prep),('rf',RandomForestClassifier(n_estimators=100,random_state=42))])
model.fit(X_tr, y_tr)
print(f'Первичная модель: {accuracy_score(y_te, model.predict(X_te)):.0%}')

## Шаг 1 · Улучши и честно оцени
Попробуй подобрать параметры и посмотри честную метрику.

In [ ]:
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.metrics import classification_report
grid = GridSearchCV(model, {'rf__n_estimators':[100,300], 'rf__max_depth':[None,5,10]}, cv=5)
grid.fit(X_tr, y_tr)
best = grid.best_estimator_
print('Лучшие параметры:', grid.best_params_)
print(classification_report(y_te, best.predict(X_te)))
print(f'Кросс-валидация: {cross_val_score(best, X, y, cv=5).mean():.1%}')

✍️ **Ответь.** Что помогло улучшить модель? Какая метрика главная для твоей задачи?

*Ответ:* …

## Шаг 2 (в Colab) · Сделай сайт через Gradio
Подставь СВОИ входы под свою задачу (у примера — признаки пингвина).

In [ ]:
!pip install gradio -q
import gradio as gr, pandas as pd
def predict(bill_len, bill_dep, flipper, mass, island):
    row = pd.DataFrame([{'bill_length_mm':bill_len,'bill_depth_mm':bill_dep,
                         'flipper_length_mm':flipper,'body_mass_g':mass,'island':island}])
    return str(best.predict(row)[0])
gr.Interface(fn=predict,
    inputs=[gr.Number(label='Длина клюва, мм'), gr.Number(label='Глубина клюва, мм'),
            gr.Number(label='Длина ласта, мм'), gr.Number(label='Масса, г'),
            gr.Radio(['Biscoe','Dream','Torgersen'], label='Остров')],
    outputs=gr.Label(label='Вид пингвина'), title='Мой проект').launch(share=True)

## Шаг 3 · README и слайды ✍️
Заполни README и собери 3–5 слайдов.

**README (заполни):**
```markdown
# <название>
## Задача
...
## Данные
...
## Модель
...
## Результаты
...
## Как запустить
...
```

**Слайды:** задача → данные → модель и метрика → демо → выводы.

## ✅ Чек-лист к уроку 23
- [ ] Модель улучшена и честно оценена (метрика + сравнение с baseline)
- [ ] Работает Gradio-интерфейс
- [ ] Заполнен README
- [ ] Готовы 3–5 слайдов

---
### 🎉 Почти готово!
Дальше — репетиция презентации и финальный деплой (урок 23), затем Demo Day.